# Predicting the **goal difference** with a Gaussian — a teaching notebook

A deliberately simple, end-to-end Bayesian workflow. Instead of modelling each
team's goals with a Poisson (what the main project does), we model **one number
per match — the goal difference** — with a **Gaussian**, then show how to fold in
**per-player squad data** and how it changes the forecasts.

### What you'll learn
1. Turn raw results into a **signed target** where the *sign is the result*.
2. Write a small **Bayesian model** as a generative story (likelihood + priors).
3. **Fit** it with MCMC and check the fit.
4. **Read** the parameters (team strengths, home advantage).
5. **Use the posterior to predict new games.**
6. **Incorporate per-player data as a prior** — step by step — and see it change
   both the team strengths and the predictions.

**Why the goal difference?**  $\text{goal\_diff} = \text{home\_goals} - \text{away\_goals}$.
The *sign* is the result: $>0$ home won, $=0$ draw, $<0$ away won.

**Why a Gaussian?** The margin is signed, roughly symmetric, bell-shaped around a
small positive number (home edge) — the simplest honest choice, and it makes the
model a clean Bayesian linear model.

In [ ]:
# --- make the wc2026 package importable under any kernel ---
import sys, pathlib
try:
    import wc2026
except ModuleNotFoundError:
    here = pathlib.Path.cwd()
    for _p in [here, *here.parents]:
        if (_p / 'src' / 'wc2026').is_dir():
            sys.path.insert(0, str(_p / 'src')); break

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import pymc as pm
import arviz as az

from wc2026.data.sources import build_real_matches
from wc2026.config import PROCESSED

RNG = 2026
plt.rcParams["figure.figsize"] = (8, 4)

## 1. Read the data
Real international results (2022 → today), filtered to the 48 World Cup teams.

In [ ]:
matches = build_real_matches(start="2022-01-01", end="2026-07-11")
matches = matches[["date", "home_team", "away_team", "home_goals", "away_goals"]].copy()
print(f"{len(matches)} matches, {matches['home_team'].nunique()} home teams")
matches.head()

## 2. The target variable: goal difference
Watch the **sign** — the mass just right of zero is the home advantage.

In [ ]:
matches["goal_diff"] = matches["home_goals"] - matches["away_goals"]
print(matches["goal_diff"].describe().round(2))
ax = matches["goal_diff"].plot.hist(bins=range(-8, 9), edgecolor="white")
ax.axvline(0, color="k", lw=1, ls="--")
ax.set_xlabel("goal difference  (home - away)")
ax.set_title("Target: signed goal difference  (<0 = away won,  >0 = home won)")
plt.show()

**Read the shape.** Roughly symmetric and bell-shaped, centred a little above
zero — exactly what a **Gaussian** describes well.

## 3. Encode teams as integers
The model refers to teams by an index.

In [ ]:
teams = sorted(set(matches["home_team"]) | set(matches["away_team"]))
team_idx = {t: i for i, t in enumerate(teams)}
n_teams = len(teams)
home_i = matches["home_team"].map(team_idx).to_numpy()
away_i = matches["away_team"].map(team_idx).to_numpy()
y = matches["goal_diff"].to_numpy().astype(float)
print(f"{n_teams} teams, {len(y)} observations")

## 4. The Bayesian model (the heart of the notebook)

Generative story: give every team one latent **strength** $s_t$. When $h$ hosts
$a$, the expected margin is the strength gap plus **home advantage** $\alpha$; the
real margin scatters by $\sigma$.

$$\text{goal\_diff}_g \sim \mathcal{N}(\mu_g,\ \sigma),\qquad
  \mu_g = \alpha + s_{h(g)} - s_{a(g)}$$
$$s_t \sim \mathcal{N}(0,\tau),\quad \alpha \sim \mathcal{N}(0.3,0.5),\quad
  \sigma \sim \text{HalfNormal}(3),\quad \tau \sim \text{HalfNormal}(1)$$

Only the *difference* of strengths matters; $s_t\sim\mathcal N(0,\tau)$ is partial
pooling that pulls thin-record teams toward average. **Note the prior mean of every
team's strength is 0 — the model knows nothing about squads yet.**

In [ ]:
with pm.Model() as model_flat:
    tau      = pm.HalfNormal("tau", 1.0)
    strength = pm.Normal("strength", 0.0, tau, shape=n_teams)
    home_adv = pm.Normal("home_adv", 0.3, 0.5)
    sigma    = pm.HalfNormal("sigma", 3.0)
    mu = home_adv + strength[home_i] - strength[away_i]
    pm.Normal("obs", mu=mu, sigma=sigma, observed=y)
    idata_flat = pm.sample(1000, tune=1000, chains=4, target_accept=0.9,
                           random_seed=RNG)

## 5. Did it converge?
**R-hat** should be $\approx 1.00$ with no divergences.

In [ ]:
az.summary(idata_flat, var_names=["home_adv", "sigma", "tau"]).round(3)

## 6. Read the parameters
Interpretable numbers: `home_adv` (goals of edge), `sigma` (match noise), and one
`strength` per team.

In [ ]:
strength_flat = idata_flat.posterior["strength"].mean(("chain", "draw")).to_numpy()
ranking = (pd.DataFrame({"team": teams, "strength": strength_flat})
           .sort_values("strength", ascending=False).reset_index(drop=True))
top = ranking.head(12).iloc[::-1]
plt.barh(top["team"], top["strength"], color="#2b8cbe")
plt.xlabel("posterior mean strength"); plt.title("Team strength (results only)")
plt.tight_layout(); plt.show()
ranking.head(10)

## 7. Using the posterior to predict **new** games

The fit gave a **posterior**: thousands of plausible parameter sets. To forecast a
match we push the *whole cloud* through the model. Because we rate *teams*, we can
price **any** pairing — even one never played.

A forecast has **two** sources of uncertainty:
1. **Parameter uncertainty** — unsure of strengths → the *expected* margin
   $\mu^{(k)} = \alpha^{(k)} + s_h^{(k)} - s_a^{(k)}$ is itself a distribution.
2. **Match noise** — even with fixed strengths, one match scatters by $\sigma$.

The **posterior-predictive** margin combines both:
$\widetilde{\text{diff}}^{(k)} = \mu^{(k)} + \sigma^{(k)}\varepsilon^{(k)}$.
Its **sign** gives win/draw/loss. The helper takes the *fitted model* as its first
argument, so later we can reuse it for the player-informed fit.

In [ ]:
def predict_new_game(idata, home, away, neutral=True, seed=0):
    """Return (mu, diff): mu = expected margin per draw (parameter uncertainty),
       diff = posterior-predictive margin (parameter uncertainty + match noise)."""
    post = idata.posterior
    st = post["strength"].stack(s=("chain", "draw")).to_numpy()   # (n_teams, S)
    ha = post["home_adv"].stack(s=("chain", "draw")).to_numpy()   # (S,)
    sg = post["sigma"].stack(s=("chain", "draw")).to_numpy()      # (S,)
    mu = (0.0 if neutral else ha) + st[team_idx[home]] - st[team_idx[away]]
    rng = np.random.default_rng(seed)
    return mu, mu + sg * rng.standard_normal(mu.shape)


def outcome_probs(diff):
    return {
        "E[margin]":  round(float(diff.mean()), 2),
        "P(home win)": round(float((diff > 0.5).mean()), 2),
        "P(draw)":     round(float((np.abs(diff) <= 0.5).mean()), 2),
        "P(away win)": round(float((diff < -0.5).mean()), 2),
    }


# The two sources of uncertainty, side by side, for one fixture:
mu, diff = predict_new_game(idata_flat, "Mexico", "South Korea", neutral=True)
plt.hist(mu,   bins=40, density=True, alpha=0.8, color="#d94801",
         label="expected margin mu (parameter uncertainty)")
plt.hist(diff, bins=40, density=True, alpha=0.5, color="#9ecae1",
         label="predictive margin (mu + match noise)")
plt.axvline(0, color="k", ls="--", lw=1)
plt.xlabel("goal difference (home - away)"); plt.legend()
plt.title("Two sources of uncertainty: Mexico vs South Korea"); plt.tight_layout(); plt.show()
print("outcome probabilities:", outcome_probs(diff))

The predictive (blue) is **much wider** than the expected-margin (orange):
the orange width is *"how unsure are we about strengths?"*, the extra blue width is
*"football is noisy."* A good forecast reports the wide one. Now a whole slate:

In [ ]:
fixtures = [("Mexico", "South Korea"), ("Brazil", "Argentina"),
            ("Spain", "France"), ("Morocco", "Portugal")]
rows = [{"home": h, "away": a, **outcome_probs(predict_new_game(idata_flat, h, a)[1])}
        for h, a in fixtures]
pd.DataFrame(rows)

## 8. Incorporating the per-player data (as a prior)

Until now `strength` started at **0** for every team — the model saw only results.
Now we fold in the **per-player squad data** the way the production model does: use
it to set each team's **prior mean**, and let results refine it.

**The single change** is the strength line — from
$$s_t \sim \mathcal{N}(0,\ \tau)
\qquad\text{to}\qquad
s_t = \underbrace{\beta_{\text{prior}}\cdot \text{prior\_strength}_t}_{\text{squad quality}}
    + \underbrace{\tau\,\tilde s_t}_{\text{residual from results}},\ \ \tilde s_t\sim\mathcal N(0,1)$$

`prior_strength` is one number per team, built by the pipeline from FIFA player
attributes (skill index, seniority, age×role). **`beta_prior` is estimated** — the
model *learns how much to trust* the squad data. We go step by step.

### Step 1 — load the player feature table
The pipeline already aggregated per-player attributes into one row per team. Let's
look at where `prior_strength` comes from.

In [ ]:
feat = pd.read_csv(PROCESSED / "team_features_real.csv")
print("columns:", list(feat.columns))
feat[["team", "squad_overall", "skill_index", "prior_strength"]].sort_values(
    "prior_strength", ascending=False).head(5)

### Step 2 — align the prior to *our* team order
Our model indexes teams in a specific order (`teams`). We must line up
`prior_strength` to the **same** order, and handle any team the feature table is
missing (fill with a neutral 0 = "no squad information, average team").

In [ ]:
prior_raw = feat.set_index("team")["prior_strength"].reindex(teams)
missing = prior_raw[prior_raw.isna()].index.tolist()
print(f"teams missing a prior ({len(missing)}): {missing}  -> filled with 0")
prior_raw = prior_raw.fillna(0.0).to_numpy()
print("aligned length:", len(prior_raw), "== n_teams:", len(prior_raw) == n_teams)

### Step 3 — standardize the prior
We put the prior on a **standard scale** (mean 0, sd 1) so it is comparable with
the residual term and `beta_prior` is easy to interpret.

In [ ]:
prior_z = (prior_raw - prior_raw.mean()) / prior_raw.std()
print(f"after standardizing:  mean={prior_z.mean():.3f}  sd={prior_z.std():.3f}")
pd.DataFrame({"team": teams, "prior_z": prior_z}).sort_values(
    "prior_z", ascending=False).head(5).round(2)

### Step 4 — build the player-informed model
Everything is identical to Section 4 **except** the `strength` line. We use the
**non-centered** form (`tau * s_raw`) for reliable sampling. Read the strength line
as: *"start at the squad-quality prior, then let the data add a residual."*

In [ ]:
with pm.Model() as model_prior:
    beta_prior = pm.Normal("beta_prior", 0.0, 1.0)      # how much to trust squads
    tau        = pm.HalfNormal("tau", 1.0)
    s_raw      = pm.Normal("s_raw", 0.0, 1.0, shape=n_teams)

    strength = pm.Deterministic("strength", beta_prior * prior_z + tau * s_raw)

    home_adv = pm.Normal("home_adv", 0.3, 0.5)
    sigma    = pm.HalfNormal("sigma", 3.0)
    mu = home_adv + strength[home_i] - strength[away_i]
    pm.Normal("obs", mu=mu, sigma=sigma, observed=y)

    idata_prior = pm.sample(1000, tune=1000, chains=4, target_accept=0.9,
                            random_seed=RNG)

### Step 5 — how much did the player data help? Read `beta_prior`
If `beta_prior`'s 94% interval sits **clear of 0**, squad quality carries real
signal for the goal difference — and the number says *how much* the model leans on
it.

In [ ]:
az.summary(idata_prior, var_names=["beta_prior", "tau", "home_adv", "sigma"]).round(3)

### Step 6 — the prior sets the start, the data refines it
Plot each team's **player-data prior** against its **learned strength**. Points near
the diagonal means the two agree; points off it are teams the *results* moved away
from their squad-only estimate.

In [ ]:
strength_prior = idata_prior.posterior["strength"].mean(("chain", "draw")).to_numpy()
comp = pd.DataFrame({"team": teams, "prior_z": prior_z, "learned": strength_prior})

plt.scatter(comp["prior_z"], comp["learned"], s=18, color="#2b8cbe")
lim = [comp[["prior_z", "learned"]].min().min() - 0.2,
       comp[["prior_z", "learned"]].max().max() + 0.2]
plt.plot(lim, lim, "k--", lw=1, alpha=0.5, label="prior = learned")
for t in ["France", "Germany", "Brazil", "Argentina", "Mexico", "United States"]:
    if t in set(comp["team"]):
        r = comp[comp["team"] == t].iloc[0]
        plt.annotate(t, (r["prior_z"], r["learned"]), fontsize=8,
                     xytext=(3, 3), textcoords="offset points")
plt.xlabel("player-data prior (standardized prior_strength)")
plt.ylabel("learned strength (posterior mean)")
plt.title("Player prior vs. what the data learned")
plt.legend(); plt.tight_layout(); plt.show()
print(f"correlation(prior, learned) = {np.corrcoef(comp['prior_z'], comp['learned'])[0,1]:.2f}")

### Step 7 — how the player data changes the **predictions**
This is the payoff. We predict the **same fixtures with both fitted models** —
`idata_flat` (results only) and `idata_prior` (results + player data) — using the
*same* `predict_new_game` helper, and compare. Player data should matter most for
teams with **few recent games**, where results alone are thin.

In [ ]:
def compare_forecast(home, away, neutral=True):
    _, d_flat  = predict_new_game(idata_flat,  home, away, neutral=neutral)
    _, d_prior = predict_new_game(idata_prior, home, away, neutral=neutral)
    f, p = outcome_probs(d_flat), outcome_probs(d_prior)
    return {
        "fixture": f"{home} v {away}",
        "margin (flat)":  f["E[margin]"],  "margin (player)": p["E[margin]"],
        "P(home) flat":   f["P(home win)"], "P(home) player":  p["P(home win)"],
    }

demo = [("Mexico", "South Korea"), ("Brazil", "Argentina"),
        ("Curaçao", "Brazil"),     ("United States", "England")]
pd.DataFrame([compare_forecast(h, a) for h, a in demo])

**Read the comparison.** Where the two `margin` columns differ, the player
data has changed the call — nudging a team up or down from what its results alone
implied. The effect is largest for sides with sparse or unrepresentative recent
records, exactly where a squad-quality prior is most useful.

**What we just did (the whole idea in one line):** *player data sets where a team
starts; the match record decides where it ends up* — and `beta_prior` quantifies
the balance. This is the same mechanism the production Poisson model uses
(`att_eff = beta_prior · prior_strength + att`), built here on the simpler
goal-difference model.

## 9. Try it yourself (exercises)
1. **Home advantage.** Re-run `compare_forecast("Mexico", "South Korea",
   neutral=False)` — how many goals is home advantage worth?
2. **Thin-record team.** Pick a team with few recent games and compare flat vs
   player forecasts. Does the prior move it more than for a data-rich team?
3. **Trust knob.** If you halved `beta_prior`'s prior sd (`pm.Normal("beta_prior",
   0, 0.5)`), would the player data influence strengths more or less?
4. **Decompose uncertainty.** For one fixture print `mu.std()` vs `diff.std()`
   (from `predict_new_game`). Which is bigger, and what does each mean?

## 10. Takeaways — and how this differs from the scoreline model

We built a one-line Bayesian idea — *margin = home edge + strength gap + noise* —
that targets the **signed goal difference**, uses a **Gaussian**, gives full
win/draw/loss probabilities from the **posterior-predictive**, and folds in
**per-player data** as a learned prior.

| | This notebook (Gaussian on the margin) | Main model (Poisson on goals) |
|---|---|---|
| Target | goal **difference** (one signed number) | each team's **goals** (two counts) |
| Gives | margin, win/draw/loss | full **scoreline** grid → 1X2, over/under, BTTS |
| Player data | prior on **strength** (Section 8) | prior on **attack** (`att_eff`) |
| Pros | dead simple, interpretable | respects that goals are counts; richer outputs |

**Prefer this one** for teaching, a quick baseline, or when you only need the
margin/result. **Prefer Poisson** for scorelines, exact totals, or simulating a
whole tournament.